In [ ]:
# ============================================================
# STEP 1: SETUP + DOWNLOAD HAM10000 + BASIC DATA VERIFICATION
# ============================================================

# Install required packages
!pip install -q kaggle

import os
import zipfile
import glob
import shutil
from pathlib import Path

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Configure Kaggle API
# ------------------------------------------------------------
# Configure Kaggle credentials outside this notebook before running.
# See: https://github.com/Kaggle/kaggle-api#api-credentials
assert os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"), (
    "Set KAGGLE_USERNAME and KAGGLE_KEY in the runtime environment."
)

# ------------------------------------------------------------
# 2. Download HAM10000 dataset
# ------------------------------------------------------------
DATA_DIR = Path("/content/HAM10000")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading dataset...")
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/HAM10000 --unzip

print("Dataset downloaded and extracted.")

# ------------------------------------------------------------
# 3. Inspect dataset files
# ------------------------------------------------------------
print("\nFiles/folders inside dataset directory:")
for item in sorted(DATA_DIR.iterdir()):
    print("-", item.name)

# ------------------------------------------------------------
# 4. Load metadata
# ------------------------------------------------------------
metadata_path = DATA_DIR / "HAM10000_metadata.csv"

if not metadata_path.exists():
    raise FileNotFoundError("HAM10000_metadata.csv not found. Check dataset extraction.")

df = pd.read_csv(metadata_path)

print("\nMetadata shape:", df.shape)
print("\nMetadata columns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

# ------------------------------------------------------------
# 5. Check classification target
# ------------------------------------------------------------
print("\nClassification target: dx")
print("Number of classes:", df["dx"].nunique())
print("\nClass distribution:")
print(df["dx"].value_counts())

# ------------------------------------------------------------
# 6. Check regression target
# ------------------------------------------------------------
print("\nRegression target: age")
print("Age missing values:", df["age"].isna().sum())
print("\nAge summary:")
print(df["age"].describe())

# ------------------------------------------------------------
# 7. Map image_id to real image paths
# ------------------------------------------------------------
image_paths = glob.glob(str(DATA_DIR / "**" / "*.jpg"), recursive=True)

image_path_dict = {
    Path(path).stem: path
    for path in image_paths
}

df["image_path"] = df["image_id"].map(image_path_dict)

missing_images = df["image_path"].isna().sum()

print("\nTotal image files found:", len(image_paths))
print("Missing image paths in metadata:", missing_images)

print("\nSample image paths:")
display(df[["image_id", "dx", "age", "image_path"]].head())

# ------------------------------------------------------------
# 8. Final Step 1 status
# ------------------------------------------------------------
if missing_images == 0:
    print("\nSTEP 1 COMPLETED SUCCESSFULLY.")
    print("Dataset is ready for Step 2: cleaning, label encoding, and train/validation/test split.")
else:
    print("\nSTEP 1 WARNING: Some images are missing. We need to fix paths before continuing.")

In [ ]:
# ============================================================
# STEP 2: CLEANING + LABEL ENCODING + LEAKAGE-FREE SPLIT
# ============================================================

import os
import glob
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold

# ------------------------------------------------------------
# 1. Reload metadata and image paths
# ------------------------------------------------------------
DATA_DIR = Path("/content/HAM10000")
metadata_path = DATA_DIR / "HAM10000_metadata.csv"

df = pd.read_csv(metadata_path)

image_paths = glob.glob(str(DATA_DIR / "**" / "*.jpg"), recursive=True)

image_path_dict = {
    Path(path).stem: path
    for path in image_paths
}

df["image_path"] = df["image_id"].map(image_path_dict)

print("Original dataset shape:", df.shape)
print("Missing image paths:", df["image_path"].isna().sum())
print("Missing age values:", df["age"].isna().sum())

# ------------------------------------------------------------
# 2. Remove rows with missing image path or missing regression target
# ------------------------------------------------------------
df = df.dropna(subset=["image_path", "age"]).reset_index(drop=True)

print("\nAfter removing missing image path / missing age:")
print("Clean dataset shape:", df.shape)
print("Missing image paths:", df["image_path"].isna().sum())
print("Missing age values:", df["age"].isna().sum())

# ------------------------------------------------------------
# 3. Encode classification labels
# ------------------------------------------------------------
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["dx"])

class_names = list(label_encoder.classes_)
num_classes = len(class_names)

print("\nClass label mapping:")
for idx, cls in enumerate(class_names):
    print(f"{idx}: {cls}")

print("\nNumber of classes:", num_classes)

# ------------------------------------------------------------
# 4. Define regression target
# ------------------------------------------------------------
df["age_target"] = df["age"].astype("float32")

print("\nRegression target summary:")
print(df["age_target"].describe())

# ------------------------------------------------------------
# 5. Leakage-free split using lesion_id as group
# ------------------------------------------------------------
# Why lesion_id?
# Some lesions have multiple images. If same lesion appears in train and test,
# the model may memorize lesion-specific patterns. So we keep the same lesion_id
# inside only one split.

df["split"] = "none"

sgkf = StratifiedGroupKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

X = df["image_path"]
y = df["label"]
groups = df["lesion_id"]

folds = list(sgkf.split(X, y, groups))

# 10 folds = approximately 10% each
test_idx = folds[0][1]
val_idx = folds[1][1]

df.loc[test_idx, "split"] = "test"
df.loc[val_idx, "split"] = "val"
df.loc[df["split"] == "none", "split"] = "train"

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

print("\nSplit sizes:")
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# ------------------------------------------------------------
# 6. Check class distribution in each split
# ------------------------------------------------------------
print("\nTrain class distribution:")
print(train_df["dx"].value_counts())

print("\nValidation class distribution:")
print(val_df["dx"].value_counts())

print("\nTest class distribution:")
print(test_df["dx"].value_counts())

# ------------------------------------------------------------
# 7. Check age distribution in each split
# ------------------------------------------------------------
print("\nAge distribution by split:")
print(df.groupby("split")["age_target"].describe())

# ------------------------------------------------------------
# 8. Confirm no lesion leakage across splits
# ------------------------------------------------------------
train_lesions = set(train_df["lesion_id"])
val_lesions   = set(val_df["lesion_id"])
test_lesions  = set(test_df["lesion_id"])

train_val_overlap = train_lesions.intersection(val_lesions)
train_test_overlap = train_lesions.intersection(test_lesions)
val_test_overlap = val_lesions.intersection(test_lesions)

print("\nLeakage check using lesion_id:")
print("Train-Val overlap :", len(train_val_overlap))
print("Train-Test overlap:", len(train_test_overlap))
print("Val-Test overlap  :", len(val_test_overlap))

# ------------------------------------------------------------
# 9. Save split CSV files for reproducibility
# ------------------------------------------------------------
SPLIT_DIR = Path("/content/ham10000_splits")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
val_df.to_csv(SPLIT_DIR / "val.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)

df.to_csv(SPLIT_DIR / "ham10000_clean_with_splits.csv", index=False)

print("\nSaved files:")
print(SPLIT_DIR / "train.csv")
print(SPLIT_DIR / "val.csv")
print(SPLIT_DIR / "test.csv")
print(SPLIT_DIR / "ham10000_clean_with_splits.csv")

# ------------------------------------------------------------
# 10. Final Step 2 status
# ------------------------------------------------------------
if (
    len(train_val_overlap) == 0 and
    len(train_test_overlap) == 0 and
    len(val_test_overlap) == 0
):
    print("\nSTEP 2 COMPLETED SUCCESSFULLY.")
    print("Clean leakage-free train/validation/test split is ready.")
    print("Next Step: build PyTorch Dataset and DataLoader for classification and regression.")
else:
    print("\nSTEP 2 WARNING: Lesion leakage detected. We need to fix the split.")

In [ ]:
# ============================================================
# STEP 3: PYTORCH DATASET + TRANSFORMS + DATALOADERS
# ============================================================

import os
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# 1. Reproducibility setup
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# ------------------------------------------------------------
# 2. Load split CSV files
# ------------------------------------------------------------
SPLIT_DIR = Path("/content/ham10000_splits")

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")

print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test_df.shape)

# ------------------------------------------------------------
# 3. Class names and number of classes
# ------------------------------------------------------------
class_names = sorted(train_df["dx"].unique())
num_classes = len(class_names)

print("\nClass names:")
print(class_names)
print("Number of classes:", num_classes)

# ------------------------------------------------------------
# 4. Age normalization for regression
# ------------------------------------------------------------
# We normalize age using TRAIN SET ONLY to avoid data leakage.
# Model predicts normalized age.
# Later, we convert predictions back to real age for MAE/RMSE.

age_mean = train_df["age_target"].mean()
age_std  = train_df["age_target"].std()

print("\nAge normalization values from TRAIN only:")
print("Age mean:", age_mean)
print("Age std :", age_std)

train_df["age_norm"] = (train_df["age_target"] - age_mean) / age_std
val_df["age_norm"]   = (val_df["age_target"] - age_mean) / age_std
test_df["age_norm"]  = (test_df["age_target"] - age_mean) / age_std

print("\nNormalized train age summary:")
print(train_df["age_norm"].describe())

# ------------------------------------------------------------
# 5. Image transforms
# ------------------------------------------------------------
# ImageNet normalization is used because VGG and ResNet pretrained models
# expect ImageNet-style input.

IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("\nTransforms created successfully.")

# ------------------------------------------------------------
# 6. Custom Dataset class
# ------------------------------------------------------------
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["image_path"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        class_label = torch.tensor(row["label"], dtype=torch.long)
        age_norm = torch.tensor(row["age_norm"], dtype=torch.float32)
        age_real = torch.tensor(row["age_target"], dtype=torch.float32)

        return {
            "image": image,
            "class_label": class_label,
            "age_norm": age_norm,
            "age_real": age_real,
            "image_id": row["image_id"],
            "dx": row["dx"]
        }

print("Dataset class created successfully.")

# ------------------------------------------------------------
# 7. Create Dataset objects
# ------------------------------------------------------------
train_dataset = HAM10000Dataset(train_df, transform=train_transform)
val_dataset   = HAM10000Dataset(val_df, transform=eval_transform)
test_dataset  = HAM10000Dataset(test_df, transform=eval_transform)

print("\nDataset sizes:")
print("Train dataset:", len(train_dataset))
print("Val dataset  :", len(val_dataset))
print("Test dataset :", len(test_dataset))

# ------------------------------------------------------------
# 8. Create DataLoaders
# ------------------------------------------------------------
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("\nDataLoaders created successfully.")
print("Batch size:", BATCH_SIZE)

# ------------------------------------------------------------
# 9. Compute class weights for imbalanced classification
# ------------------------------------------------------------
# HAM10000 is highly imbalanced, especially nv vs df/vasc.
# We will use these weights in CrossEntropyLoss later.

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_df["label"].values
)

class_weights = torch.tensor(class_weights_np, dtype=torch.float32).to(device)

print("\nClass weights:")
for cls_name, weight in zip(class_names, class_weights_np):
    print(f"{cls_name}: {weight:.4f}")

# ------------------------------------------------------------
# 10. Test one batch
# ------------------------------------------------------------
batch = next(iter(train_loader))

images = batch["image"]
class_labels = batch["class_label"]
age_norm = batch["age_norm"]
age_real = batch["age_real"]

print("\nOne training batch check:")
print("Image batch shape      :", images.shape)
print("Class label shape      :", class_labels.shape)
print("Normalized age shape   :", age_norm.shape)
print("Real age shape         :", age_real.shape)
print("Image dtype            :", images.dtype)
print("Class label dtype      :", class_labels.dtype)
print("Age dtype              :", age_norm.dtype)

print("\nSample labels from batch:")
print("Class labels:", class_labels[:10].tolist())
print("Real ages   :", age_real[:10].tolist())
print("Norm ages   :", age_norm[:10].tolist())

# ------------------------------------------------------------
# 11. Save configuration for later report/model loading
# ------------------------------------------------------------
CONFIG_DIR = Path("/content/ham10000_config")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "num_classes": num_classes,
    "class_names": class_names,
    "age_mean": float(age_mean),
    "age_std": float(age_std),
    "classification_target": "dx",
    "regression_target": "age",
    "train_size": len(train_dataset),
    "val_size": len(val_dataset),
    "test_size": len(test_dataset)
}

with open(CONFIG_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=4)

np.save(CONFIG_DIR / "class_weights.npy", class_weights_np)

print("\nSaved config files:")
print(CONFIG_DIR / "config.json")
print(CONFIG_DIR / "class_weights.npy")

# ------------------------------------------------------------
# 12. Final Step 3 status
# ------------------------------------------------------------
print("\nSTEP 3 COMPLETED SUCCESSFULLY.")
print("PyTorch Dataset and DataLoaders are ready.")
print("Next Step: build VGG16 classification model.")

In [ ]:
# ============================================================
# STEP 4: VGG16 CLASSIFICATION MODEL + TRAINING + EVALUATION
# ============================================================

import os
import time
import copy
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.models as models
from torchvision.models import VGG16_Weights

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# ------------------------------------------------------------
# 1. Output directories
# ------------------------------------------------------------
MODEL_DIR = Path("/content/ham10000_models")
RESULT_DIR = Path("/content/ham10000_results")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Build VGG16 classification model
# ------------------------------------------------------------
def build_vgg16_classifier(num_classes):
    weights = VGG16_Weights.IMAGENET1K_V1
    model = models.vgg16(weights=weights)

    # Freeze early feature extractor layers
    for param in model.features.parameters():
        param.requires_grad = False

    # Fine-tune only the last VGG convolution block
    for param in model.features[24:].parameters():
        param.requires_grad = True

    # Replace classifier head for HAM10000 7-class classification
    model.classifier = nn.Sequential(
        nn.Linear(25088, 1024),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(1024, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )

    return model

vgg16_cls_model = build_vgg16_classifier(num_classes).to(device)

print("VGG16 classification model created successfully.")

# ------------------------------------------------------------
# 3. Count trainable and total parameters
# ------------------------------------------------------------
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

total_params, trainable_params = count_parameters(vgg16_cls_model)

print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# ------------------------------------------------------------
# 4. Loss function, optimizer, scheduler
# ------------------------------------------------------------
criterion_cls = nn.CrossEntropyLoss(weight=class_weights)

optimizer_cls = optim.AdamW(
    filter(lambda p: p.requires_grad, vgg16_cls_model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler_cls = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_cls,
    mode="max",
    factor=0.5,
    patience=2
)

print("Loss, optimizer, and scheduler are ready.")

# ------------------------------------------------------------
# 5. Training and validation functions
# ------------------------------------------------------------
def train_one_epoch_classification(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in dataloader:
        images = batch["image"].to(device)
        labels = batch["class_label"].to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1


def evaluate_classification(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device)
            labels = batch["class_label"].to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1, all_labels, all_preds

print("Training and evaluation functions are ready.")

# ------------------------------------------------------------
# 6. Train VGG16 classification model
# ------------------------------------------------------------
EPOCHS = 8
best_val_f1 = 0.0
best_model_wts = copy.deepcopy(vgg16_cls_model.state_dict())

history_vgg16_cls = {
    "train_loss": [],
    "train_acc": [],
    "train_f1": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": []
}

start_time = time.time()

print("\nStarting VGG16 classification training...\n")

for epoch in range(EPOCHS):
    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print("-" * 50)

    train_loss, train_acc, train_f1 = train_one_epoch_classification(
        vgg16_cls_model,
        train_loader,
        criterion_cls,
        optimizer_cls,
        device
    )

    val_loss, val_acc, val_f1, val_labels, val_preds = evaluate_classification(
        vgg16_cls_model,
        val_loader,
        criterion_cls,
        device
    )

    scheduler_cls.step(val_f1)

    history_vgg16_cls["train_loss"].append(train_loss)
    history_vgg16_cls["train_acc"].append(train_acc)
    history_vgg16_cls["train_f1"].append(train_f1)
    history_vgg16_cls["val_loss"].append(val_loss)
    history_vgg16_cls["val_acc"].append(val_acc)
    history_vgg16_cls["val_f1"].append(val_f1)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Macro F1: {train_f1:.4f}")
    print(f"Val Loss  : {val_loss:.4f} | Val Acc  : {val_acc:.4f} | Val Macro F1  : {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_wts = copy.deepcopy(vgg16_cls_model.state_dict())

        torch.save(
            best_model_wts,
            MODEL_DIR / "vgg16_classification_best.pth"
        )

        print("Best model saved.")

    print()

training_time = time.time() - start_time

print(f"Training completed in {training_time/60:.2f} minutes.")
print(f"Best validation Macro F1: {best_val_f1:.4f}")

# Load best model weights
vgg16_cls_model.load_state_dict(best_model_wts)

# ------------------------------------------------------------
# 7. Evaluate best model on test set
# ------------------------------------------------------------
test_loss, test_acc, test_f1, test_labels, test_preds = evaluate_classification(
    vgg16_cls_model,
    test_loader,
    criterion_cls,
    device
)

print("\nVGG16 Classification Test Results")
print("-" * 50)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Macro F1 : {test_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        test_labels,
        test_preds,
        target_names=class_names,
        digits=4
    )
)

# ------------------------------------------------------------
# 8. Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(test_labels, test_preds)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(9, 8))
disp.plot(
    ax=ax,
    xticks_rotation=45,
    values_format="d"
)
plt.title("VGG16 Classification - Confusion Matrix")
plt.tight_layout()
plt.savefig(RESULT_DIR / "vgg16_classification_confusion_matrix.png", dpi=300)
plt.show()

print("Confusion matrix saved to:")
print(RESULT_DIR / "vgg16_classification_confusion_matrix.png")

# ------------------------------------------------------------
# 9. Save history and test metrics
# ------------------------------------------------------------
history_df = pd.DataFrame(history_vgg16_cls)
history_df.to_csv(RESULT_DIR / "vgg16_classification_history.csv", index=False)

vgg16_cls_metrics = {
    "model": "VGG16",
    "task": "Classification",
    "test_loss": test_loss,
    "test_accuracy": test_acc,
    "test_macro_f1": test_f1,
    "best_val_macro_f1": best_val_f1,
    "training_time_minutes": training_time / 60,
    "total_parameters": total_params,
    "trainable_parameters": trainable_params
}

metrics_df = pd.DataFrame([vgg16_cls_metrics])
metrics_df.to_csv(RESULT_DIR / "vgg16_classification_metrics.csv", index=False)

print("\nSaved files:")
print(MODEL_DIR / "vgg16_classification_best.pth")
print(RESULT_DIR / "vgg16_classification_history.csv")
print(RESULT_DIR / "vgg16_classification_metrics.csv")

# ------------------------------------------------------------
# 10. Final Step 4 status
# ------------------------------------------------------------
print("\nSTEP 4 COMPLETED SUCCESSFULLY.")
print("VGG16 classification model is trained, evaluated, and saved.")
print("Next Step: VGG16 regression model for age prediction.")

In [ ]:
# ============================================================
# STEP 5: VGG16 REGRESSION MODEL + TRAINING + EVALUATION
# Target: age
# Output: Linear scalar
# Loss: MSE
# ============================================================

import os
import time
import copy
import gc
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.models as models
from torchvision.models import VGG16_Weights

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Clear GPU memory from previous model
# ------------------------------------------------------------
try:
    del vgg16_cls_model
    del optimizer_cls
    del criterion_cls
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared for VGG16 regression.")

# ------------------------------------------------------------
# 2. Output directories
# ------------------------------------------------------------
MODEL_DIR = Path("/content/ham10000_models")
RESULT_DIR = Path("/content/ham10000_results")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Build VGG16 regression model
# ------------------------------------------------------------
def build_vgg16_regressor():
    weights = VGG16_Weights.IMAGENET1K_V1
    model = models.vgg16(weights=weights)

    # Freeze early feature layers
    for param in model.features.parameters():
        param.requires_grad = False

    # Fine-tune last VGG convolution block
    for param in model.features[24:].parameters():
        param.requires_grad = True

    # Replace classifier with regression head
    # Final output is one linear scalar value for normalized age
    model.classifier = nn.Sequential(
        nn.Linear(25088, 1024),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(1024, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, 1)
    )

    return model

vgg16_reg_model = build_vgg16_regressor().to(device)

print("VGG16 regression model created successfully.")

# ------------------------------------------------------------
# 4. Count trainable and total parameters
# ------------------------------------------------------------
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

total_params_reg, trainable_params_reg = count_parameters(vgg16_reg_model)

print(f"Total parameters    : {total_params_reg:,}")
print(f"Trainable parameters: {trainable_params_reg:,}")

# ------------------------------------------------------------
# 5. Loss function, optimizer, scheduler
# ------------------------------------------------------------
criterion_reg = nn.MSELoss()

optimizer_reg = optim.AdamW(
    filter(lambda p: p.requires_grad, vgg16_reg_model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler_reg = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_reg,
    mode="min",
    factor=0.5,
    patience=2
)

print("Loss, optimizer, and scheduler are ready.")

# ------------------------------------------------------------
# 6. Helper function: convert normalized age back to real age
# ------------------------------------------------------------
def denormalize_age(age_norm_values):
    return (age_norm_values * age_std) + age_mean

# ------------------------------------------------------------
# 7. Training and validation functions
# ------------------------------------------------------------
def train_one_epoch_regression(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_preds_norm = []
    all_targets_norm = []

    for batch in dataloader:
        images = batch["image"].to(device)
        targets = batch["age_norm"].to(device).view(-1, 1)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        all_preds_norm.extend(outputs.detach().cpu().numpy().flatten())
        all_targets_norm.extend(targets.detach().cpu().numpy().flatten())

    epoch_loss = running_loss / len(dataloader.dataset)

    preds_real = denormalize_age(np.array(all_preds_norm))
    targets_real = denormalize_age(np.array(all_targets_norm))

    mae = mean_absolute_error(targets_real, preds_real)
    mse = mean_squared_error(targets_real, preds_real)
    rmse = np.sqrt(mse)
    r2 = r2_score(targets_real, preds_real)

    return epoch_loss, mae, rmse, r2


def evaluate_regression(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds_norm = []
    all_targets_norm = []
    all_targets_real_direct = []

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device)
            targets = batch["age_norm"].to(device).view(-1, 1)
            age_real = batch["age_real"].cpu().numpy().flatten()

            outputs = model(images)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * images.size(0)

            all_preds_norm.extend(outputs.detach().cpu().numpy().flatten())
            all_targets_norm.extend(targets.detach().cpu().numpy().flatten())
            all_targets_real_direct.extend(age_real)

    epoch_loss = running_loss / len(dataloader.dataset)

    preds_real = denormalize_age(np.array(all_preds_norm))
    targets_real = np.array(all_targets_real_direct)

    mae = mean_absolute_error(targets_real, preds_real)
    mse = mean_squared_error(targets_real, preds_real)
    rmse = np.sqrt(mse)
    r2 = r2_score(targets_real, preds_real)

    return epoch_loss, mae, rmse, r2, targets_real, preds_real

print("Training and evaluation functions are ready.")

# ------------------------------------------------------------
# 8. Train VGG16 regression model
# ------------------------------------------------------------
EPOCHS = 8
best_val_mae = float("inf")
best_model_wts = copy.deepcopy(vgg16_reg_model.state_dict())

history_vgg16_reg = {
    "train_loss_mse_norm": [],
    "train_mae_age": [],
    "train_rmse_age": [],
    "train_r2": [],
    "val_loss_mse_norm": [],
    "val_mae_age": [],
    "val_rmse_age": [],
    "val_r2": []
}

start_time = time.time()

print("\nStarting VGG16 regression training...\n")

for epoch in range(EPOCHS):
    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print("-" * 50)

    train_loss, train_mae, train_rmse, train_r2 = train_one_epoch_regression(
        vgg16_reg_model,
        train_loader,
        criterion_reg,
        optimizer_reg,
        device
    )

    val_loss, val_mae, val_rmse, val_r2, val_targets, val_preds = evaluate_regression(
        vgg16_reg_model,
        val_loader,
        criterion_reg,
        device
    )

    scheduler_reg.step(val_mae)

    history_vgg16_reg["train_loss_mse_norm"].append(train_loss)
    history_vgg16_reg["train_mae_age"].append(train_mae)
    history_vgg16_reg["train_rmse_age"].append(train_rmse)
    history_vgg16_reg["train_r2"].append(train_r2)

    history_vgg16_reg["val_loss_mse_norm"].append(val_loss)
    history_vgg16_reg["val_mae_age"].append(val_mae)
    history_vgg16_reg["val_rmse_age"].append(val_rmse)
    history_vgg16_reg["val_r2"].append(val_r2)

    print(f"Train Loss(MSE norm): {train_loss:.4f} | Train MAE(age): {train_mae:.2f} | Train RMSE(age): {train_rmse:.2f} | Train R2: {train_r2:.4f}")
    print(f"Val Loss(MSE norm)  : {val_loss:.4f} | Val MAE(age)  : {val_mae:.2f} | Val RMSE(age)  : {val_rmse:.2f} | Val R2  : {val_r2:.4f}")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_model_wts = copy.deepcopy(vgg16_reg_model.state_dict())

        torch.save(
            best_model_wts,
            MODEL_DIR / "vgg16_regression_best.pth"
        )

        print("Best model saved.")

    print()

training_time_reg = time.time() - start_time

print(f"Training completed in {training_time_reg/60:.2f} minutes.")
print(f"Best validation MAE(age): {best_val_mae:.2f}")

# Load best model weights
vgg16_reg_model.load_state_dict(best_model_wts)

# ------------------------------------------------------------
# 9. Evaluate best model on test set
# ------------------------------------------------------------
test_loss, test_mae, test_rmse, test_r2, test_targets, test_preds = evaluate_regression(
    vgg16_reg_model,
    test_loader,
    criterion_reg,
    device
)

print("\nVGG16 Regression Test Results")
print("-" * 50)
print(f"Test Loss(MSE normalized): {test_loss:.4f}")
print(f"Test MAE(age)            : {test_mae:.2f}")
print(f"Test RMSE(age)           : {test_rmse:.2f}")
print(f"Test R2                  : {test_r2:.4f}")

# ------------------------------------------------------------
# 10. Plot predicted age vs true age
# ------------------------------------------------------------
plt.figure(figsize=(7, 7))
plt.scatter(test_targets, test_preds, alpha=0.5)
plt.plot(
    [test_targets.min(), test_targets.max()],
    [test_targets.min(), test_targets.max()],
    linestyle="--"
)
plt.xlabel("True Age")
plt.ylabel("Predicted Age")
plt.title("VGG16 Regression - True Age vs Predicted Age")
plt.tight_layout()
plt.savefig(RESULT_DIR / "vgg16_regression_true_vs_predicted.png", dpi=300)
plt.show()

print("Regression scatter plot saved to:")
print(RESULT_DIR / "vgg16_regression_true_vs_predicted.png")

# ------------------------------------------------------------
# 11. Save predictions
# ------------------------------------------------------------
vgg16_reg_predictions_df = pd.DataFrame({
    "true_age": test_targets,
    "predicted_age": test_preds,
    "absolute_error": np.abs(test_targets - test_preds)
})

vgg16_reg_predictions_df.to_csv(
    RESULT_DIR / "vgg16_regression_test_predictions.csv",
    index=False
)

# ------------------------------------------------------------
# 12. Save history and test metrics
# ------------------------------------------------------------
history_df = pd.DataFrame(history_vgg16_reg)
history_df.to_csv(RESULT_DIR / "vgg16_regression_history.csv", index=False)

vgg16_reg_metrics = {
    "model": "VGG16",
    "task": "Regression",
    "test_loss_mse_normalized": test_loss,
    "test_mae_age": test_mae,
    "test_rmse_age": test_rmse,
    "test_r2": test_r2,
    "best_val_mae_age": best_val_mae,
    "training_time_minutes": training_time_reg / 60,
    "total_parameters": total_params_reg,
    "trainable_parameters": trainable_params_reg
}

metrics_df = pd.DataFrame([vgg16_reg_metrics])
metrics_df.to_csv(RESULT_DIR / "vgg16_regression_metrics.csv", index=False)

print("\nSaved files:")
print(MODEL_DIR / "vgg16_regression_best.pth")
print(RESULT_DIR / "vgg16_regression_history.csv")
print(RESULT_DIR / "vgg16_regression_metrics.csv")
print(RESULT_DIR / "vgg16_regression_test_predictions.csv")

# ------------------------------------------------------------
# 13. Final Step 5 status
# ------------------------------------------------------------
print("\nSTEP 5 COMPLETED SUCCESSFULLY.")
print("VGG16 regression model is trained, evaluated, and saved.")
print("Next Step: ResNet50 classification model.")

In [ ]:
# ============================================================
# STEP 6: RESNET50 CLASSIFICATION MODEL + TRAINING + EVALUATION
# Task: 7-class skin lesion classification
# Loss: Cross-Entropy
# Metrics: Accuracy, Macro F1, Confusion Matrix
# ============================================================

import os
import time
import copy
import gc
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.models as models
from torchvision.models import ResNet50_Weights

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# ------------------------------------------------------------
# 1. Clear GPU memory from previous model
# ------------------------------------------------------------
try:
    del vgg16_reg_model
    del optimizer_reg
    del criterion_reg
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared for ResNet50 classification.")

# ------------------------------------------------------------
# 2. Output directories
# ------------------------------------------------------------
MODEL_DIR = Path("/content/ham10000_models")
RESULT_DIR = Path("/content/ham10000_results")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Build ResNet50 classification model
# ------------------------------------------------------------
def build_resnet50_classifier(num_classes):
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)

    # Freeze all layers first
    for param in model.parameters():
        param.requires_grad = False

    # Fine-tune the final residual block
    for param in model.layer4.parameters():
        param.requires_grad = True

    # Replace final fully connected layer
    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(512, 128),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(128, num_classes)
    )

    return model

resnet50_cls_model = build_resnet50_classifier(num_classes).to(device)

print("ResNet50 classification model created successfully.")

# ------------------------------------------------------------
# 4. Count trainable and total parameters
# ------------------------------------------------------------
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

total_params_resnet_cls, trainable_params_resnet_cls = count_parameters(resnet50_cls_model)

print(f"Total parameters    : {total_params_resnet_cls:,}")
print(f"Trainable parameters: {trainable_params_resnet_cls:,}")

# ------------------------------------------------------------
# 5. Loss function, optimizer, scheduler
# ------------------------------------------------------------
criterion_resnet_cls = nn.CrossEntropyLoss(weight=class_weights)

optimizer_resnet_cls = optim.AdamW(
    filter(lambda p: p.requires_grad, resnet50_cls_model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler_resnet_cls = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_resnet_cls,
    mode="max",
    factor=0.5,
    patience=2
)

print("Loss, optimizer, and scheduler are ready.")

# ------------------------------------------------------------
# 6. Training and evaluation functions
# ------------------------------------------------------------
def train_one_epoch_classification(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in dataloader:
        images = batch["image"].to(device)
        labels = batch["class_label"].to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1


def evaluate_classification(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device)
            labels = batch["class_label"].to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1, all_labels, all_preds

print("Training and evaluation functions are ready.")

# ------------------------------------------------------------
# 7. Train ResNet50 classification model
# ------------------------------------------------------------
EPOCHS = 8
best_val_f1 = 0.0
best_model_wts = copy.deepcopy(resnet50_cls_model.state_dict())

history_resnet50_cls = {
    "train_loss": [],
    "train_acc": [],
    "train_f1": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": []
}

start_time = time.time()

print("\nStarting ResNet50 classification training...\n")

for epoch in range(EPOCHS):
    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print("-" * 50)

    train_loss, train_acc, train_f1 = train_one_epoch_classification(
        resnet50_cls_model,
        train_loader,
        criterion_resnet_cls,
        optimizer_resnet_cls,
        device
    )

    val_loss, val_acc, val_f1, val_labels, val_preds = evaluate_classification(
        resnet50_cls_model,
        val_loader,
        criterion_resnet_cls,
        device
    )

    scheduler_resnet_cls.step(val_f1)

    history_resnet50_cls["train_loss"].append(train_loss)
    history_resnet50_cls["train_acc"].append(train_acc)
    history_resnet50_cls["train_f1"].append(train_f1)
    history_resnet50_cls["val_loss"].append(val_loss)
    history_resnet50_cls["val_acc"].append(val_acc)
    history_resnet50_cls["val_f1"].append(val_f1)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Macro F1: {train_f1:.4f}")
    print(f"Val Loss  : {val_loss:.4f} | Val Acc  : {val_acc:.4f} | Val Macro F1  : {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_wts = copy.deepcopy(resnet50_cls_model.state_dict())

        torch.save(
            best_model_wts,
            MODEL_DIR / "resnet50_classification_best.pth"
        )

        print("Best model saved.")

    print()

training_time_resnet_cls = time.time() - start_time

print(f"Training completed in {training_time_resnet_cls/60:.2f} minutes.")
print(f"Best validation Macro F1: {best_val_f1:.4f}")

# Load best model weights
resnet50_cls_model.load_state_dict(best_model_wts)

# ------------------------------------------------------------
# 8. Evaluate best model on test set
# ------------------------------------------------------------
test_loss, test_acc, test_f1, test_labels, test_preds = evaluate_classification(
    resnet50_cls_model,
    test_loader,
    criterion_resnet_cls,
    device
)

print("\nResNet50 Classification Test Results")
print("-" * 50)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Macro F1 : {test_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        test_labels,
        test_preds,
        target_names=class_names,
        digits=4
    )
)

# ------------------------------------------------------------
# 9. Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(test_labels, test_preds)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(9, 8))
disp.plot(
    ax=ax,
    xticks_rotation=45,
    values_format="d"
)
plt.title("ResNet50 Classification - Confusion Matrix")
plt.tight_layout()
plt.savefig(RESULT_DIR / "resnet50_classification_confusion_matrix.png", dpi=300)
plt.show()

print("Confusion matrix saved to:")
print(RESULT_DIR / "resnet50_classification_confusion_matrix.png")

# ------------------------------------------------------------
# 10. Save history and test metrics
# ------------------------------------------------------------
history_df = pd.DataFrame(history_resnet50_cls)
history_df.to_csv(RESULT_DIR / "resnet50_classification_history.csv", index=False)

resnet50_cls_metrics = {
    "model": "ResNet50",
    "task": "Classification",
    "test_loss": test_loss,
    "test_accuracy": test_acc,
    "test_macro_f1": test_f1,
    "best_val_macro_f1": best_val_f1,
    "training_time_minutes": training_time_resnet_cls / 60,
    "total_parameters": total_params_resnet_cls,
    "trainable_parameters": trainable_params_resnet_cls
}

metrics_df = pd.DataFrame([resnet50_cls_metrics])
metrics_df.to_csv(RESULT_DIR / "resnet50_classification_metrics.csv", index=False)

print("\nSaved files:")
print(MODEL_DIR / "resnet50_classification_best.pth")
print(RESULT_DIR / "resnet50_classification_history.csv")
print(RESULT_DIR / "resnet50_classification_metrics.csv")
print(RESULT_DIR / "resnet50_classification_confusion_matrix.png")

# ------------------------------------------------------------
# 11. Final Step 6 status
# ------------------------------------------------------------
print("\nSTEP 6 COMPLETED SUCCESSFULLY.")
print("ResNet50 classification model is trained, evaluated, and saved.")
print("Next Step: ResNet50 regression model for age prediction.")

In [ ]:
# ============================================================
# STEP 7: RESNET50 REGRESSION MODEL + TRAINING + EVALUATION
# Target: age
# Output: Linear scalar
# Loss: MSE
# Metrics: MAE, RMSE, R2
# ============================================================

import os
import time
import copy
import gc
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.models as models
from torchvision.models import ResNet50_Weights

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Clear GPU memory from previous model
# ------------------------------------------------------------
try:
    del resnet50_cls_model
    del optimizer_resnet_cls
    del criterion_resnet_cls
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared for ResNet50 regression.")

# ------------------------------------------------------------
# 2. Output directories
# ------------------------------------------------------------
MODEL_DIR = Path("/content/ham10000_models")
RESULT_DIR = Path("/content/ham10000_results")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Build ResNet50 regression model
# ------------------------------------------------------------
def build_resnet50_regressor():
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)

    # Freeze all layers first
    for param in model.parameters():
        param.requires_grad = False

    # Fine-tune final residual block
    for param in model.layer4.parameters():
        param.requires_grad = True

    # Replace final fully connected layer with regression head
    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(512, 128),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(128, 1)
    )

    return model

resnet50_reg_model = build_resnet50_regressor().to(device)

print("ResNet50 regression model created successfully.")

# ------------------------------------------------------------
# 4. Count trainable and total parameters
# ------------------------------------------------------------
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

total_params_resnet_reg, trainable_params_resnet_reg = count_parameters(resnet50_reg_model)

print(f"Total parameters    : {total_params_resnet_reg:,}")
print(f"Trainable parameters: {trainable_params_resnet_reg:,}")

# ------------------------------------------------------------
# 5. Loss function, optimizer, scheduler
# ------------------------------------------------------------
criterion_resnet_reg = nn.MSELoss()

optimizer_resnet_reg = optim.AdamW(
    filter(lambda p: p.requires_grad, resnet50_reg_model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler_resnet_reg = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_resnet_reg,
    mode="min",
    factor=0.5,
    patience=2
)

print("Loss, optimizer, and scheduler are ready.")

# ------------------------------------------------------------
# 6. Helper function: convert normalized age back to real age
# ------------------------------------------------------------
def denormalize_age(age_norm_values):
    return (age_norm_values * age_std) + age_mean

# ------------------------------------------------------------
# 7. Training and evaluation functions
# ------------------------------------------------------------
def train_one_epoch_regression(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_preds_norm = []
    all_targets_norm = []

    for batch in dataloader:
        images = batch["image"].to(device)
        targets = batch["age_norm"].to(device).view(-1, 1)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        all_preds_norm.extend(outputs.detach().cpu().numpy().flatten())
        all_targets_norm.extend(targets.detach().cpu().numpy().flatten())

    epoch_loss = running_loss / len(dataloader.dataset)

    preds_real = denormalize_age(np.array(all_preds_norm))
    targets_real = denormalize_age(np.array(all_targets_norm))

    mae = mean_absolute_error(targets_real, preds_real)
    mse = mean_squared_error(targets_real, preds_real)
    rmse = np.sqrt(mse)
    r2 = r2_score(targets_real, preds_real)

    return epoch_loss, mae, rmse, r2


def evaluate_regression(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds_norm = []
    all_targets_real_direct = []

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device)
            targets = batch["age_norm"].to(device).view(-1, 1)
            age_real = batch["age_real"].cpu().numpy().flatten()

            outputs = model(images)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * images.size(0)

            all_preds_norm.extend(outputs.detach().cpu().numpy().flatten())
            all_targets_real_direct.extend(age_real)

    epoch_loss = running_loss / len(dataloader.dataset)

    preds_real = denormalize_age(np.array(all_preds_norm))
    targets_real = np.array(all_targets_real_direct)

    mae = mean_absolute_error(targets_real, preds_real)
    mse = mean_squared_error(targets_real, preds_real)
    rmse = np.sqrt(mse)
    r2 = r2_score(targets_real, preds_real)

    return epoch_loss, mae, rmse, r2, targets_real, preds_real

print("Training and evaluation functions are ready.")

# ------------------------------------------------------------
# 8. Train ResNet50 regression model
# ------------------------------------------------------------
EPOCHS = 8
best_val_mae = float("inf")
best_model_wts = copy.deepcopy(resnet50_reg_model.state_dict())

history_resnet50_reg = {
    "train_loss_mse_norm": [],
    "train_mae_age": [],
    "train_rmse_age": [],
    "train_r2": [],
    "val_loss_mse_norm": [],
    "val_mae_age": [],
    "val_rmse_age": [],
    "val_r2": []
}

start_time = time.time()

print("\nStarting ResNet50 regression training...\n")

for epoch in range(EPOCHS):
    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print("-" * 50)

    train_loss, train_mae, train_rmse, train_r2 = train_one_epoch_regression(
        resnet50_reg_model,
        train_loader,
        criterion_resnet_reg,
        optimizer_resnet_reg,
        device
    )

    val_loss, val_mae, val_rmse, val_r2, val_targets, val_preds = evaluate_regression(
        resnet50_reg_model,
        val_loader,
        criterion_resnet_reg,
        device
    )

    scheduler_resnet_reg.step(val_mae)

    history_resnet50_reg["train_loss_mse_norm"].append(train_loss)
    history_resnet50_reg["train_mae_age"].append(train_mae)
    history_resnet50_reg["train_rmse_age"].append(train_rmse)
    history_resnet50_reg["train_r2"].append(train_r2)

    history_resnet50_reg["val_loss_mse_norm"].append(val_loss)
    history_resnet50_reg["val_mae_age"].append(val_mae)
    history_resnet50_reg["val_rmse_age"].append(val_rmse)
    history_resnet50_reg["val_r2"].append(val_r2)

    print(f"Train Loss(MSE norm): {train_loss:.4f} | Train MAE(age): {train_mae:.2f} | Train RMSE(age): {train_rmse:.2f} | Train R2: {train_r2:.4f}")
    print(f"Val Loss(MSE norm)  : {val_loss:.4f} | Val MAE(age)  : {val_mae:.2f} | Val RMSE(age)  : {val_rmse:.2f} | Val R2  : {val_r2:.4f}")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_model_wts = copy.deepcopy(resnet50_reg_model.state_dict())

        torch.save(
            best_model_wts,
            MODEL_DIR / "resnet50_regression_best.pth"
        )

        print("Best model saved.")

    print()

training_time_resnet_reg = time.time() - start_time

print(f"Training completed in {training_time_resnet_reg/60:.2f} minutes.")
print(f"Best validation MAE(age): {best_val_mae:.2f}")

# Load best model weights
resnet50_reg_model.load_state_dict(best_model_wts)

# ------------------------------------------------------------
# 9. Evaluate best model on test set
# ------------------------------------------------------------
test_loss, test_mae, test_rmse, test_r2, test_targets, test_preds = evaluate_regression(
    resnet50_reg_model,
    test_loader,
    criterion_resnet_reg,
    device
)

print("\nResNet50 Regression Test Results")
print("-" * 50)
print(f"Test Loss(MSE normalized): {test_loss:.4f}")
print(f"Test MAE(age)            : {test_mae:.2f}")
print(f"Test RMSE(age)           : {test_rmse:.2f}")
print(f"Test R2                  : {test_r2:.4f}")

# ------------------------------------------------------------
# 10. Plot predicted age vs true age
# ------------------------------------------------------------
plt.figure(figsize=(7, 7))
plt.scatter(test_targets, test_preds, alpha=0.5)
plt.plot(
    [test_targets.min(), test_targets.max()],
    [test_targets.min(), test_targets.max()],
    linestyle="--"
)
plt.xlabel("True Age")
plt.ylabel("Predicted Age")
plt.title("ResNet50 Regression - True Age vs Predicted Age")
plt.tight_layout()
plt.savefig(RESULT_DIR / "resnet50_regression_true_vs_predicted.png", dpi=300)
plt.show()

print("Regression scatter plot saved to:")
print(RESULT_DIR / "resnet50_regression_true_vs_predicted.png")

# ------------------------------------------------------------
# 11. Save predictions
# ------------------------------------------------------------
resnet50_reg_predictions_df = pd.DataFrame({
    "true_age": test_targets,
    "predicted_age": test_preds,
    "absolute_error": np.abs(test_targets - test_preds)
})

resnet50_reg_predictions_df.to_csv(
    RESULT_DIR / "resnet50_regression_test_predictions.csv",
    index=False
)

# ------------------------------------------------------------
# 12. Save history and test metrics
# ------------------------------------------------------------
history_df = pd.DataFrame(history_resnet50_reg)
history_df.to_csv(RESULT_DIR / "resnet50_regression_history.csv", index=False)

resnet50_reg_metrics = {
    "model": "ResNet50",
    "task": "Regression",
    "test_loss_mse_normalized": test_loss,
    "test_mae_age": test_mae,
    "test_rmse_age": test_rmse,
    "test_r2": test_r2,
    "best_val_mae_age": best_val_mae,
    "training_time_minutes": training_time_resnet_reg / 60,
    "total_parameters": total_params_resnet_reg,
    "trainable_parameters": trainable_params_resnet_reg
}

metrics_df = pd.DataFrame([resnet50_reg_metrics])
metrics_df.to_csv(RESULT_DIR / "resnet50_regression_metrics.csv", index=False)

print("\nSaved files:")
print(MODEL_DIR / "resnet50_regression_best.pth")
print(RESULT_DIR / "resnet50_regression_history.csv")
print(RESULT_DIR / "resnet50_regression_metrics.csv")
print(RESULT_DIR / "resnet50_regression_test_predictions.csv")
print(RESULT_DIR / "resnet50_regression_true_vs_predicted.png")

# ------------------------------------------------------------
# 13. Final Step 7 status
# ------------------------------------------------------------
print("\nSTEP 7 COMPLETED SUCCESSFULLY.")
print("ResNet50 regression model is trained, evaluated, and saved.")
print("Next Step: Grad-CAM visualizations for all four models.")

In [ ]:
# ============================================================
# STEP 8 FIXED: GRAD-CAM VISUALIZATIONS FOR ALL FOUR MODELS
# Fix: Disable in-place ReLU to avoid backward-hook error
# ============================================================

import os
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import VGG16_Weights, ResNet50_Weights

import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ------------------------------------------------------------
# 1. Clean previous failed Grad-CAM objects
# ------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_DIR = Path("/content/ham10000_models")
RESULT_DIR = Path("/content/ham10000_results")
GRADCAM_DIR = RESULT_DIR / "gradcam_visualizations"

GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

print("Using device:", device)
print("Grad-CAM output directory:", GRADCAM_DIR)

# ------------------------------------------------------------
# 2. Load config and test dataframe
# ------------------------------------------------------------
CONFIG_DIR = Path("/content/ham10000_config")
SPLIT_DIR = Path("/content/ham10000_splits")

with open(CONFIG_DIR / "config.json", "r") as f:
    config = json.load(f)

class_names = config["class_names"]
num_classes = config["num_classes"]
age_mean = config["age_mean"]
age_std = config["age_std"]
IMG_SIZE = config["img_size"]

test_df = pd.read_csv(SPLIT_DIR / "test.csv")

print("\nLoaded config:")
print("Class names:", class_names)
print("Number of classes:", num_classes)
print("Test dataframe shape:", test_df.shape)

# ------------------------------------------------------------
# 3. Transform and helper functions
# ------------------------------------------------------------
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def load_image_for_gradcam(image_path):
    image_pil = Image.open(image_path).convert("RGB")
    image_tensor = eval_transform(image_pil).unsqueeze(0)
    return image_pil, image_tensor

def tensor_to_display_image(image_tensor):
    img = image_tensor.squeeze(0).detach().cpu().numpy()
    img = np.transpose(img, (1, 2, 0))

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    img = (img * std) + mean
    img = np.clip(img, 0, 1)

    return img

def overlay_cam_on_image(rgb_img, cam_map, alpha=0.45):
    heatmap = cm.jet(cam_map)[:, :, :3]
    overlay = (1 - alpha) * rgb_img + alpha * heatmap
    overlay = np.clip(overlay, 0, 1)
    return overlay

def denormalize_age(age_norm_value):
    return (age_norm_value * age_std) + age_mean

# ------------------------------------------------------------
# 4. IMPORTANT FIX: disable in-place ReLU
# ------------------------------------------------------------
def disable_inplace_relu(model):
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False
    return model

# ------------------------------------------------------------
# 5. Rebuild model architectures exactly as trained
# ------------------------------------------------------------
def build_vgg16_classifier(num_classes):
    model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)

    for param in model.features.parameters():
        param.requires_grad = False

    for param in model.features[24:].parameters():
        param.requires_grad = True

    model.classifier = nn.Sequential(
        nn.Linear(25088, 1024),
        nn.ReLU(inplace=False),
        nn.Dropout(0.5),
        nn.Linear(1024, 256),
        nn.ReLU(inplace=False),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )

    model = disable_inplace_relu(model)
    return model

def build_vgg16_regressor():
    model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)

    for param in model.features.parameters():
        param.requires_grad = False

    for param in model.features[24:].parameters():
        param.requires_grad = True

    model.classifier = nn.Sequential(
        nn.Linear(25088, 1024),
        nn.ReLU(inplace=False),
        nn.Dropout(0.5),
        nn.Linear(1024, 256),
        nn.ReLU(inplace=False),
        nn.Dropout(0.3),
        nn.Linear(256, 1)
    )

    model = disable_inplace_relu(model)
    return model

def build_resnet50_classifier(num_classes):
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

    for param in model.parameters():
        param.requires_grad = False

    for param in model.layer4.parameters():
        param.requires_grad = True

    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=False),
        nn.Dropout(0.4),
        nn.Linear(512, 128),
        nn.ReLU(inplace=False),
        nn.Dropout(0.3),
        nn.Linear(128, num_classes)
    )

    model = disable_inplace_relu(model)
    return model

def build_resnet50_regressor():
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

    for param in model.parameters():
        param.requires_grad = False

    for param in model.layer4.parameters():
        param.requires_grad = True

    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=False),
        nn.Dropout(0.4),
        nn.Linear(512, 128),
        nn.ReLU(inplace=False),
        nn.Dropout(0.3),
        nn.Linear(128, 1)
    )

    model = disable_inplace_relu(model)
    return model

print("\nModel builders are ready with in-place ReLU disabled.")

# ------------------------------------------------------------
# 6. Grad-CAM class
# ------------------------------------------------------------
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_hook = self.target_layer.register_forward_hook(self.save_activation)
        self.backward_hook = self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output.detach().clone()

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach().clone()

    def generate(self, input_tensor, task_type="classification", target_class=None):
        self.model.zero_grad(set_to_none=True)

        output = self.model(input_tensor)

        if task_type == "classification":
            if target_class is None:
                target_class = torch.argmax(output, dim=1).item()
            score = output[:, target_class]

        elif task_type == "regression":
            score = output[:, 0]

        else:
            raise ValueError("task_type must be classification or regression.")

        score.backward()

        gradients = self.gradients
        activations = self.activations

        weights = torch.mean(gradients, dim=(2, 3), keepdim=True)
        cam_map = torch.sum(weights * activations, dim=1)

        cam_map = torch.relu(cam_map)
        cam_map = cam_map.squeeze().detach().cpu().numpy()

        cam_map = cam_map - np.min(cam_map)

        if np.max(cam_map) != 0:
            cam_map = cam_map / np.max(cam_map)

        cam_map = Image.fromarray(np.uint8(cam_map * 255))
        cam_map = cam_map.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        cam_map = np.array(cam_map).astype(np.float32) / 255.0

        return cam_map, output.detach()

    def remove_hooks(self):
        self.forward_hook.remove()
        self.backward_hook.remove()

print("Grad-CAM class created successfully.")

# ------------------------------------------------------------
# 7. Select representative samples
# ------------------------------------------------------------
selected_rows = []

for cls in class_names:
    cls_rows = test_df[test_df["dx"] == cls]
    if len(cls_rows) > 0:
        selected_rows.append(cls_rows.iloc[0])

selected_df = pd.DataFrame(selected_rows).reset_index(drop=True)

# Use 4 classes to keep each figure readable
selected_df = selected_df.head(4)

print("\nSelected samples for Grad-CAM:")
display(selected_df[["image_id", "dx", "age_target", "image_path"]])

# ------------------------------------------------------------
# 8. Function to run Grad-CAM for one model
# ------------------------------------------------------------
def run_gradcam_for_model(
    model_name,
    task_type,
    model_builder,
    weights_path,
    target_layer_getter,
    selected_df
):
    print("\n" + "=" * 70)
    print(f"Running Grad-CAM for: {model_name} | Task: {task_type}")
    print("=" * 70)

    gc.collect()
    torch.cuda.empty_cache()

    model = model_builder().to(device)
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()

    target_layer = target_layer_getter(model)
    gradcam = GradCAM(model, target_layer)

    fig, axes = plt.subplots(
        len(selected_df),
        3,
        figsize=(12, 4 * len(selected_df))
    )

    if len(selected_df) == 1:
        axes = np.expand_dims(axes, axis=0)

    interpretation_records = []

    for i, row in selected_df.iterrows():
        image_pil, image_tensor = load_image_for_gradcam(row["image_path"])
        image_tensor = image_tensor.to(device)

        cam_map, output = gradcam.generate(
            image_tensor,
            task_type=task_type
        )

        display_img = tensor_to_display_image(image_tensor)
        overlay = overlay_cam_on_image(display_img, cam_map)

        if task_type == "classification":
            probs = torch.softmax(output, dim=1).cpu().numpy().flatten()
            pred_idx = int(np.argmax(probs))
            pred_label = class_names[pred_idx]
            confidence = float(probs[pred_idx])

            title_text = (
                f"True: {row['dx']} | Pred: {pred_label}\n"
                f"Confidence: {confidence:.3f}"
            )

            interpretation_records.append({
                "model": model_name,
                "task": task_type,
                "image_id": row["image_id"],
                "true_class": row["dx"],
                "predicted_class": pred_label,
                "confidence": confidence,
                "true_age": row["age_target"],
                "predicted_age": None
            })

        else:
            pred_age_norm = output.cpu().numpy().flatten()[0]
            pred_age = denormalize_age(pred_age_norm)

            title_text = (
                f"True age: {row['age_target']:.1f}\n"
                f"Pred age: {pred_age:.1f}"
            )

            interpretation_records.append({
                "model": model_name,
                "task": task_type,
                "image_id": row["image_id"],
                "true_class": row["dx"],
                "predicted_class": None,
                "confidence": None,
                "true_age": row["age_target"],
                "predicted_age": float(pred_age)
            })

        axes[i, 0].imshow(display_img)
        axes[i, 0].set_title("Original Image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(cam_map)
        axes[i, 1].set_title("Grad-CAM Heatmap")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title(title_text)
        axes[i, 2].axis("off")

    plt.suptitle(f"{model_name} {task_type.capitalize()} Grad-CAM", fontsize=16)
    plt.tight_layout()

    save_path = GRADCAM_DIR / f"{model_name.lower()}_{task_type}_gradcam.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    gradcam.remove_hooks()

    del model
    del gradcam

    gc.collect()
    torch.cuda.empty_cache()

    print("Saved Grad-CAM figure:")
    print(save_path)

    return interpretation_records

# ------------------------------------------------------------
# 9. Run Grad-CAM for all four required models
# ------------------------------------------------------------
all_interpretation_records = []

# 1. VGG16 Classification
records = run_gradcam_for_model(
    model_name="VGG16",
    task_type="classification",
    model_builder=lambda: build_vgg16_classifier(num_classes),
    weights_path=MODEL_DIR / "vgg16_classification_best.pth",
    target_layer_getter=lambda model: model.features[28],
    selected_df=selected_df
)
all_interpretation_records.extend(records)

# 2. VGG16 Regression
records = run_gradcam_for_model(
    model_name="VGG16",
    task_type="regression",
    model_builder=build_vgg16_regressor,
    weights_path=MODEL_DIR / "vgg16_regression_best.pth",
    target_layer_getter=lambda model: model.features[28],
    selected_df=selected_df
)
all_interpretation_records.extend(records)

# 3. ResNet50 Classification
records = run_gradcam_for_model(
    model_name="ResNet50",
    task_type="classification",
    model_builder=lambda: build_resnet50_classifier(num_classes),
    weights_path=MODEL_DIR / "resnet50_classification_best.pth",
    target_layer_getter=lambda model: model.layer4[-1].conv3,
    selected_df=selected_df
)
all_interpretation_records.extend(records)

# 4. ResNet50 Regression
records = run_gradcam_for_model(
    model_name="ResNet50",
    task_type="regression",
    model_builder=build_resnet50_regressor,
    weights_path=MODEL_DIR / "resnet50_regression_best.pth",
    target_layer_getter=lambda model: model.layer4[-1].conv3,
    selected_df=selected_df
)
all_interpretation_records.extend(records)

# ------------------------------------------------------------
# 10. Save interpretation records
# ------------------------------------------------------------
interpretation_df = pd.DataFrame(all_interpretation_records)

interpretation_csv_path = GRADCAM_DIR / "gradcam_interpretation_records.csv"
interpretation_df.to_csv(interpretation_csv_path, index=False)

print("\nGrad-CAM interpretation records:")
display(interpretation_df)

print("\nSaved interpretation table:")
print(interpretation_csv_path)

# ------------------------------------------------------------
# 11. Final Step 8 status
# ------------------------------------------------------------
print("\nSTEP 8 COMPLETED SUCCESSFULLY.")
print("Grad-CAM visualizations generated for all four models.")
print("Next Step: create final comparison table and report-ready results.")

In [ ]:
# ============================================================
# STEP 9: FINAL COMPARISON TABLE + REPORT-READY SUMMARY
# Models:
# 1. VGG16 Classification
# 2. VGG16 Regression
# 3. ResNet50 Classification
# 4. ResNet50 Regression
# ============================================================

import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
RESULT_DIR = Path("/content/ham10000_results")
GRADCAM_DIR = RESULT_DIR / "gradcam_visualizations"

print("Result directory:", RESULT_DIR)

# ------------------------------------------------------------
# 2. Load saved metric files
# ------------------------------------------------------------
vgg16_cls = pd.read_csv(RESULT_DIR / "vgg16_classification_metrics.csv")
vgg16_reg = pd.read_csv(RESULT_DIR / "vgg16_regression_metrics.csv")
resnet50_cls = pd.read_csv(RESULT_DIR / "resnet50_classification_metrics.csv")
resnet50_reg = pd.read_csv(RESULT_DIR / "resnet50_regression_metrics.csv")

print("Metric files loaded successfully.")

# ------------------------------------------------------------
# 3. Build unified comparison table
# ------------------------------------------------------------
comparison_rows = []

# Classification rows
for df_metric in [vgg16_cls, resnet50_cls]:
    row = df_metric.iloc[0]

    comparison_rows.append({
        "Model": row["model"],
        "Task": row["task"],
        "Test Accuracy": row["test_accuracy"],
        "Test Macro F1": row["test_macro_f1"],
        "Test MAE(age)": np.nan,
        "Test RMSE(age)": np.nan,
        "Test R2": np.nan,
        "Best Val Macro F1": row["best_val_macro_f1"],
        "Best Val MAE(age)": np.nan,
        "Training Time(min)": row["training_time_minutes"],
        "Total Parameters": int(row["total_parameters"]),
        "Trainable Parameters": int(row["trainable_parameters"])
    })

# Regression rows
for df_metric in [vgg16_reg, resnet50_reg]:
    row = df_metric.iloc[0]

    comparison_rows.append({
        "Model": row["model"],
        "Task": row["task"],
        "Test Accuracy": np.nan,
        "Test Macro F1": np.nan,
        "Test MAE(age)": row["test_mae_age"],
        "Test RMSE(age)": row["test_rmse_age"],
        "Test R2": row["test_r2"],
        "Best Val Macro F1": np.nan,
        "Best Val MAE(age)": row["best_val_mae_age"],
        "Training Time(min)": row["training_time_minutes"],
        "Total Parameters": int(row["total_parameters"]),
        "Trainable Parameters": int(row["trainable_parameters"])
    })

comparison_df = pd.DataFrame(comparison_rows)

# Sort table by model and task
comparison_df["Task_Order"] = comparison_df["Task"].map({
    "Classification": 1,
    "Regression": 2
})

comparison_df = comparison_df.sort_values(
    by=["Model", "Task_Order"]
).drop(columns=["Task_Order"]).reset_index(drop=True)

# ------------------------------------------------------------
# 4. Round numeric values for clean reporting
# ------------------------------------------------------------
report_table = comparison_df.copy()

round_cols = [
    "Test Accuracy",
    "Test Macro F1",
    "Test MAE(age)",
    "Test RMSE(age)",
    "Test R2",
    "Best Val Macro F1",
    "Best Val MAE(age)",
    "Training Time(min)"
]

for col in round_cols:
    report_table[col] = report_table[col].round(4)

print("\nFinal Comparison Table:")
display(report_table)

# ------------------------------------------------------------
# 5. Save comparison table
# ------------------------------------------------------------
comparison_csv_path = RESULT_DIR / "final_model_comparison_table.csv"
comparison_excel_path = RESULT_DIR / "final_model_comparison_table.xlsx"

report_table.to_csv(comparison_csv_path, index=False)
report_table.to_excel(comparison_excel_path, index=False)

print("\nSaved comparison tables:")
print(comparison_csv_path)
print(comparison_excel_path)

# ------------------------------------------------------------
# 6. Classification comparison plot
# ------------------------------------------------------------
classification_df = report_table[report_table["Task"] == "Classification"].copy()

plt.figure(figsize=(8, 5))
x = np.arange(len(classification_df))
width = 0.35

plt.bar(
    x - width / 2,
    classification_df["Test Accuracy"],
    width,
    label="Accuracy"
)

plt.bar(
    x + width / 2,
    classification_df["Test Macro F1"],
    width,
    label="Macro F1"
)

plt.xticks(x, classification_df["Model"])
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Classification Performance Comparison")
plt.legend()
plt.tight_layout()

classification_plot_path = RESULT_DIR / "classification_performance_comparison.png"
plt.savefig(classification_plot_path, dpi=300)
plt.show()

print("Saved classification comparison plot:")
print(classification_plot_path)

# ------------------------------------------------------------
# 7. Regression comparison plot: MAE and RMSE
# ------------------------------------------------------------
regression_df = report_table[report_table["Task"] == "Regression"].copy()

plt.figure(figsize=(8, 5))
x = np.arange(len(regression_df))
width = 0.35

plt.bar(
    x - width / 2,
    regression_df["Test MAE(age)"],
    width,
    label="MAE"
)

plt.bar(
    x + width / 2,
    regression_df["Test RMSE(age)"],
    width,
    label="RMSE"
)

plt.xticks(x, regression_df["Model"])
plt.ylabel("Error in Years")
plt.title("Regression Error Comparison")
plt.legend()
plt.tight_layout()

regression_error_plot_path = RESULT_DIR / "regression_error_comparison.png"
plt.savefig(regression_error_plot_path, dpi=300)
plt.show()

print("Saved regression error comparison plot:")
print(regression_error_plot_path)

# ------------------------------------------------------------
# 8. Regression R2 comparison plot
# ------------------------------------------------------------
plt.figure(figsize=(7, 5))

plt.bar(
    regression_df["Model"],
    regression_df["Test R2"]
)

plt.ylabel("R2 Score")
plt.title("Regression R2 Comparison")
plt.tight_layout()

regression_r2_plot_path = RESULT_DIR / "regression_r2_comparison.png"
plt.savefig(regression_r2_plot_path, dpi=300)
plt.show()

print("Saved regression R2 comparison plot:")
print(regression_r2_plot_path)

# ------------------------------------------------------------
# 9. Automatically identify best models
# ------------------------------------------------------------
best_cls_row = classification_df.loc[
    classification_df["Test Macro F1"].idxmax()
]

best_reg_row = regression_df.loc[
    regression_df["Test MAE(age)"].idxmin()
]

print("\nBest Classification Model:")
print("Model:", best_cls_row["Model"])
print("Test Accuracy:", best_cls_row["Test Accuracy"])
print("Test Macro F1:", best_cls_row["Test Macro F1"])

print("\nBest Regression Model:")
print("Model:", best_reg_row["Model"])
print("Test MAE(age):", best_reg_row["Test MAE(age)"])
print("Test RMSE(age):", best_reg_row["Test RMSE(age)"])
print("Test R2:", best_reg_row["Test R2"])

# ------------------------------------------------------------
# 10. Check required Grad-CAM files
# ------------------------------------------------------------
required_gradcam_files = [
    GRADCAM_DIR / "vgg16_classification_gradcam.png",
    GRADCAM_DIR / "vgg16_regression_gradcam.png",
    GRADCAM_DIR / "resnet50_classification_gradcam.png",
    GRADCAM_DIR / "resnet50_regression_gradcam.png",
    GRADCAM_DIR / "gradcam_interpretation_records.csv"
]

print("\nGrad-CAM File Check:")
for file_path in required_gradcam_files:
    print(file_path.name, "FOUND" if file_path.exists() else "MISSING")

# ------------------------------------------------------------
# 11. Create report-ready text summary
# ------------------------------------------------------------
summary_text = f"""
FINAL RESULT SUMMARY

Dataset:
HAM10000 skin lesion dataset was used for both classification and regression.
Classification target: dx
Regression target: age

Classification Results:
VGG16 achieved test accuracy of {classification_df[classification_df['Model'] == 'VGG16']['Test Accuracy'].values[0]:.4f}
and macro F1-score of {classification_df[classification_df['Model'] == 'VGG16']['Test Macro F1'].values[0]:.4f}.

ResNet50 achieved test accuracy of {classification_df[classification_df['Model'] == 'ResNet50']['Test Accuracy'].values[0]:.4f}
and macro F1-score of {classification_df[classification_df['Model'] == 'ResNet50']['Test Macro F1'].values[0]:.4f}.

Best classification model:
{best_cls_row['Model']} achieved the highest macro F1-score of {best_cls_row['Test Macro F1']:.4f}.

Regression Results:
VGG16 achieved test MAE of {regression_df[regression_df['Model'] == 'VGG16']['Test MAE(age)'].values[0]:.4f} years,
RMSE of {regression_df[regression_df['Model'] == 'VGG16']['Test RMSE(age)'].values[0]:.4f} years,
and R2 of {regression_df[regression_df['Model'] == 'VGG16']['Test R2'].values[0]:.4f}.

ResNet50 achieved test MAE of {regression_df[regression_df['Model'] == 'ResNet50']['Test MAE(age)'].values[0]:.4f} years,
RMSE of {regression_df[regression_df['Model'] == 'ResNet50']['Test RMSE(age)'].values[0]:.4f} years,
and R2 of {regression_df[regression_df['Model'] == 'ResNet50']['Test R2'].values[0]:.4f}.

Best regression model:
{best_reg_row['Model']} achieved the lowest MAE of {best_reg_row['Test MAE(age)']:.4f} years.

Interpretability:
Grad-CAM visualizations were generated for all four trained models:
VGG16 classification, VGG16 regression, ResNet50 classification, and ResNet50 regression.
"""

summary_path = RESULT_DIR / "final_report_ready_summary.txt"

with open(summary_path, "w") as f:
    f.write(summary_text)

print("\nReport-ready summary saved to:")
print(summary_path)

print("\nReport-ready summary:")
print(summary_text)

# ------------------------------------------------------------
# 12. Submission checklist status
# ------------------------------------------------------------
print("\nSUBMISSION CHECKLIST STATUS")
print("- Dataset source link and justification: READY")
print("- Fully executable notebook: READY if all cells from Step 1 to Step 9 are kept")
print("- Saved model weights: READY")
print("- Grad-CAM visualizations for all four models: READY")
print("- Comparison table with all metrics: READY")
print("- IEEE-format PDF report: NEXT STEP")

# ------------------------------------------------------------
# 13. Final Step 9 status
# ------------------------------------------------------------
print("\nSTEP 9 COMPLETED SUCCESSFULLY.")
print("Final comparison table and report-ready result summary are complete.")
print("Next Step: write the IEEE-format report.")

In [ ]:
# ============================================================
# TRAINING AND VALIDATION CURVES FOR VISUALIZATION
# Models:
# 1. VGG16 Classification
# 2. ResNet50 Classification
# 3. VGG16 Regression
# 4. ResNet50 Regression
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
RESULT_DIR = Path("/content/ham10000_results")
CURVE_DIR = RESULT_DIR / "training_validation_curves"
CURVE_DIR.mkdir(parents=True, exist_ok=True)

print("Curve output directory:", CURVE_DIR)

# ------------------------------------------------------------
# 2. Load history files
# ------------------------------------------------------------
vgg16_cls_history = pd.read_csv(RESULT_DIR / "vgg16_classification_history.csv")
resnet50_cls_history = pd.read_csv(RESULT_DIR / "resnet50_classification_history.csv")

vgg16_reg_history = pd.read_csv(RESULT_DIR / "vgg16_regression_history.csv")
resnet50_reg_history = pd.read_csv(RESULT_DIR / "resnet50_regression_history.csv")

print("History files loaded successfully.")

# ------------------------------------------------------------
# 3. Helper function for plotting curves
# ------------------------------------------------------------
def plot_train_val_curve(
    history_df,
    train_col,
    val_col,
    title,
    ylabel,
    save_name
):
    epochs = range(1, len(history_df) + 1)

    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs,
        history_df[train_col],
        marker="o",
        label="Training"
    )

    plt.plot(
        epochs,
        history_df[val_col],
        marker="o",
        label="Validation"
    )

    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(epochs)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    save_path = CURVE_DIR / save_name
    plt.savefig(save_path, dpi=300)
    plt.show()

    print("Saved:", save_path)

# ============================================================
# 4. VGG16 Classification Curves
# ============================================================

plot_train_val_curve(
    history_df=vgg16_cls_history,
    train_col="train_loss",
    val_col="val_loss",
    title="VGG16 Classification - Training vs Validation Loss",
    ylabel="Cross-Entropy Loss",
    save_name="vgg16_classification_loss_curve.png"
)

plot_train_val_curve(
    history_df=vgg16_cls_history,
    train_col="train_acc",
    val_col="val_acc",
    title="VGG16 Classification - Training vs Validation Accuracy",
    ylabel="Accuracy",
    save_name="vgg16_classification_accuracy_curve.png"
)

plot_train_val_curve(
    history_df=vgg16_cls_history,
    train_col="train_f1",
    val_col="val_f1",
    title="VGG16 Classification - Training vs Validation Macro F1",
    ylabel="Macro F1-score",
    save_name="vgg16_classification_f1_curve.png"
)

# ============================================================
# 5. ResNet50 Classification Curves
# ============================================================

plot_train_val_curve(
    history_df=resnet50_cls_history,
    train_col="train_loss",
    val_col="val_loss",
    title="ResNet50 Classification - Training vs Validation Loss",
    ylabel="Cross-Entropy Loss",
    save_name="resnet50_classification_loss_curve.png"
)

plot_train_val_curve(
    history_df=resnet50_cls_history,
    train_col="train_acc",
    val_col="val_acc",
    title="ResNet50 Classification - Training vs Validation Accuracy",
    ylabel="Accuracy",
    save_name="resnet50_classification_accuracy_curve.png"
)

plot_train_val_curve(
    history_df=resnet50_cls_history,
    train_col="train_f1",
    val_col="val_f1",
    title="ResNet50 Classification - Training vs Validation Macro F1",
    ylabel="Macro F1-score",
    save_name="resnet50_classification_f1_curve.png"
)

# ============================================================
# 6. VGG16 Regression Curves
# ============================================================

plot_train_val_curve(
    history_df=vgg16_reg_history,
    train_col="train_loss_mse_norm",
    val_col="val_loss_mse_norm",
    title="VGG16 Regression - Training vs Validation MSE Loss",
    ylabel="MSE Loss on Normalized Age",
    save_name="vgg16_regression_loss_curve.png"
)

plot_train_val_curve(
    history_df=vgg16_reg_history,
    train_col="train_mae_age",
    val_col="val_mae_age",
    title="VGG16 Regression - Training vs Validation MAE",
    ylabel="MAE in Years",
    save_name="vgg16_regression_mae_curve.png"
)

plot_train_val_curve(
    history_df=vgg16_reg_history,
    train_col="train_rmse_age",
    val_col="val_rmse_age",
    title="VGG16 Regression - Training vs Validation RMSE",
    ylabel="RMSE in Years",
    save_name="vgg16_regression_rmse_curve.png"
)

plot_train_val_curve(
    history_df=vgg16_reg_history,
    train_col="train_r2",
    val_col="val_r2",
    title="VGG16 Regression - Training vs Validation R2",
    ylabel="R2 Score",
    save_name="vgg16_regression_r2_curve.png"
)

# ============================================================
# 7. ResNet50 Regression Curves
# ============================================================

plot_train_val_curve(
    history_df=resnet50_reg_history,
    train_col="train_loss_mse_norm",
    val_col="val_loss_mse_norm",
    title="ResNet50 Regression - Training vs Validation MSE Loss",
    ylabel="MSE Loss on Normalized Age",
    save_name="resnet50_regression_loss_curve.png"
)

plot_train_val_curve(
    history_df=resnet50_reg_history,
    train_col="train_mae_age",
    val_col="val_mae_age",
    title="ResNet50 Regression - Training vs Validation MAE",
    ylabel="MAE in Years",
    save_name="resnet50_regression_mae_curve.png"
)

plot_train_val_curve(
    history_df=resnet50_reg_history,
    train_col="train_rmse_age",
    val_col="val_rmse_age",
    title="ResNet50 Regression - Training vs Validation RMSE",
    ylabel="RMSE in Years",
    save_name="resnet50_regression_rmse_curve.png"
)

plot_train_val_curve(
    history_df=resnet50_reg_history,
    train_col="train_r2",
    val_col="val_r2",
    title="ResNet50 Regression - Training vs Validation R2",
    ylabel="R2 Score",
    save_name="resnet50_regression_r2_curve.png"
)

# ------------------------------------------------------------
# 8. Final check
# ------------------------------------------------------------
print("\nAll training and validation curves generated successfully.")
print("Saved in:", CURVE_DIR)

print("\nGenerated files:")
for file in sorted(CURVE_DIR.glob("*.png")):
    print("-", file.name)

bonus work

In [ ]:
# ============================================================
# STEP 11: BONUS IMPLEMENTATION - GRAD-CAM++
# Extra CAM Variant for Bonus Mark
# Models:
# 1. VGG16 Classification
# 2. VGG16 Regression
# 3. ResNet50 Classification
# 4. ResNet50 Regression
# ============================================================

import os
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import VGG16_Weights, ResNet50_Weights

import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ------------------------------------------------------------
# 1. Paths and device
# ------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_DIR = Path("/content/ham10000_models")
RESULT_DIR = Path("/content/ham10000_results")
CONFIG_DIR = Path("/content/ham10000_config")
SPLIT_DIR = Path("/content/ham10000_splits")

GRADCAMPP_DIR = RESULT_DIR / "gradcampp_visualizations"
GRADCAMPP_DIR.mkdir(parents=True, exist_ok=True)

print("Using device:", device)
print("Grad-CAM++ output directory:", GRADCAMPP_DIR)

# ------------------------------------------------------------
# 2. Load config and test data
# ------------------------------------------------------------
with open(CONFIG_DIR / "config.json", "r") as f:
    config = json.load(f)

class_names = config["class_names"]
num_classes = config["num_classes"]
age_mean = config["age_mean"]
age_std = config["age_std"]
IMG_SIZE = config["img_size"]

test_df = pd.read_csv(SPLIT_DIR / "test.csv")

print("\nLoaded config:")
print("Class names:", class_names)
print("Number of classes:", num_classes)
print("Test dataframe shape:", test_df.shape)

# ------------------------------------------------------------
# 3. Image transform and helper functions
# ------------------------------------------------------------
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def load_image_for_cam(image_path):
    image_pil = Image.open(image_path).convert("RGB")
    image_tensor = eval_transform(image_pil).unsqueeze(0)
    return image_pil, image_tensor

def tensor_to_display_image(image_tensor):
    img = image_tensor.squeeze(0).detach().cpu().numpy()
    img = np.transpose(img, (1, 2, 0))

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    img = (img * std) + mean
    img = np.clip(img, 0, 1)

    return img

def overlay_cam_on_image(rgb_img, cam_map, alpha=0.45):
    heatmap = cm.jet(cam_map)[:, :, :3]
    overlay = (1 - alpha) * rgb_img + alpha * heatmap
    overlay = np.clip(overlay, 0, 1)
    return overlay

def denormalize_age(age_norm_value):
    return (age_norm_value * age_std) + age_mean

# ------------------------------------------------------------
# 4. Disable in-place ReLU
# ------------------------------------------------------------
def disable_inplace_relu(model):
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False
    return model

# ------------------------------------------------------------
# 5. Rebuild trained model architectures
# ------------------------------------------------------------
def build_vgg16_classifier(num_classes):
    model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)

    for param in model.features.parameters():
        param.requires_grad = False

    for param in model.features[24:].parameters():
        param.requires_grad = True

    model.classifier = nn.Sequential(
        nn.Linear(25088, 1024),
        nn.ReLU(inplace=False),
        nn.Dropout(0.5),
        nn.Linear(1024, 256),
        nn.ReLU(inplace=False),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )

    model = disable_inplace_relu(model)
    return model

def build_vgg16_regressor():
    model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)

    for param in model.features.parameters():
        param.requires_grad = False

    for param in model.features[24:].parameters():
        param.requires_grad = True

    model.classifier = nn.Sequential(
        nn.Linear(25088, 1024),
        nn.ReLU(inplace=False),
        nn.Dropout(0.5),
        nn.Linear(1024, 256),
        nn.ReLU(inplace=False),
        nn.Dropout(0.3),
        nn.Linear(256, 1)
    )

    model = disable_inplace_relu(model)
    return model

def build_resnet50_classifier(num_classes):
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

    for param in model.parameters():
        param.requires_grad = False

    for param in model.layer4.parameters():
        param.requires_grad = True

    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=False),
        nn.Dropout(0.4),
        nn.Linear(512, 128),
        nn.ReLU(inplace=False),
        nn.Dropout(0.3),
        nn.Linear(128, num_classes)
    )

    model = disable_inplace_relu(model)
    return model

def build_resnet50_regressor():
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

    for param in model.parameters():
        param.requires_grad = False

    for param in model.layer4.parameters():
        param.requires_grad = True

    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=False),
        nn.Dropout(0.4),
        nn.Linear(512, 128),
        nn.ReLU(inplace=False),
        nn.Dropout(0.3),
        nn.Linear(128, 1)
    )

    model = disable_inplace_relu(model)
    return model

print("\nModel builders ready.")

# ------------------------------------------------------------
# 6. Grad-CAM++ class
# ------------------------------------------------------------
class GradCAMPlusPlus:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_hook = self.target_layer.register_forward_hook(self.save_activation)
        self.backward_hook = self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output.detach().clone()

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach().clone()

    def generate(self, input_tensor, task_type="classification", target_class=None):
        self.model.zero_grad(set_to_none=True)

        output = self.model(input_tensor)

        if task_type == "classification":
            if target_class is None:
                target_class = torch.argmax(output, dim=1).item()
            score = output[:, target_class]

        elif task_type == "regression":
            score = output[:, 0]

        else:
            raise ValueError("task_type must be classification or regression.")

        score.backward(retain_graph=True)

        gradients = self.gradients
        activations = self.activations

        # Grad-CAM++ formula
        grads_power_2 = gradients ** 2
        grads_power_3 = gradients ** 3

        eps = 1e-8

        sum_activations = torch.sum(
            activations,
            dim=(2, 3),
            keepdim=True
        )

        alpha_num = grads_power_2
        alpha_denom = 2 * grads_power_2 + sum_activations * grads_power_3
        alpha_denom = torch.where(
            alpha_denom != 0.0,
            alpha_denom,
            torch.ones_like(alpha_denom) * eps
        )

        alphas = alpha_num / (alpha_denom + eps)

        positive_gradients = torch.relu(gradients)

        weights = torch.sum(
            alphas * positive_gradients,
            dim=(2, 3),
            keepdim=True
        )

        cam_map = torch.sum(weights * activations, dim=1)
        cam_map = torch.relu(cam_map)

        cam_map = cam_map.squeeze().detach().cpu().numpy()

        cam_map = cam_map - np.min(cam_map)

        if np.max(cam_map) != 0:
            cam_map = cam_map / np.max(cam_map)

        cam_map = Image.fromarray(np.uint8(cam_map * 255))
        cam_map = cam_map.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        cam_map = np.array(cam_map).astype(np.float32) / 255.0

        return cam_map, output.detach()

    def remove_hooks(self):
        self.forward_hook.remove()
        self.backward_hook.remove()

print("Grad-CAM++ class created successfully.")

# ------------------------------------------------------------
# 7. Select representative samples
# ------------------------------------------------------------
selected_rows = []

for cls in class_names:
    cls_rows = test_df[test_df["dx"] == cls]
    if len(cls_rows) > 0:
        selected_rows.append(cls_rows.iloc[0])

selected_df = pd.DataFrame(selected_rows).reset_index(drop=True)

# Use 4 samples to keep the figure readable
selected_df = selected_df.head(4)

print("\nSelected samples for Grad-CAM++:")
display(selected_df[["image_id", "dx", "age_target", "image_path"]])

# ------------------------------------------------------------
# 8. Function to run Grad-CAM++ for one model
# ------------------------------------------------------------
def run_gradcampp_for_model(
    model_name,
    task_type,
    model_builder,
    weights_path,
    target_layer_getter,
    selected_df
):
    print("\n" + "=" * 70)
    print(f"Running Grad-CAM++ for: {model_name} | Task: {task_type}")
    print("=" * 70)

    gc.collect()
    torch.cuda.empty_cache()

    model = model_builder().to(device)
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()

    target_layer = target_layer_getter(model)
    campp = GradCAMPlusPlus(model, target_layer)

    fig, axes = plt.subplots(
        len(selected_df),
        3,
        figsize=(12, 4 * len(selected_df))
    )

    if len(selected_df) == 1:
        axes = np.expand_dims(axes, axis=0)

    records = []

    for i, row in selected_df.iterrows():
        image_pil, image_tensor = load_image_for_cam(row["image_path"])
        image_tensor = image_tensor.to(device)

        cam_map, output = campp.generate(
            image_tensor,
            task_type=task_type
        )

        display_img = tensor_to_display_image(image_tensor)
        overlay = overlay_cam_on_image(display_img, cam_map)

        if task_type == "classification":
            probs = torch.softmax(output, dim=1).cpu().numpy().flatten()
            pred_idx = int(np.argmax(probs))
            pred_label = class_names[pred_idx]
            confidence = float(probs[pred_idx])

            title_text = (
                f"True: {row['dx']} | Pred: {pred_label}\n"
                f"Conf: {confidence:.3f}"
            )

            records.append({
                "cam_method": "Grad-CAM++",
                "model": model_name,
                "task": task_type,
                "image_id": row["image_id"],
                "true_class": row["dx"],
                "predicted_class": pred_label,
                "confidence": confidence,
                "true_age": row["age_target"],
                "predicted_age": None
            })

        else:
            pred_age_norm = output.cpu().numpy().flatten()[0]
            pred_age = denormalize_age(pred_age_norm)

            title_text = (
                f"True age: {row['age_target']:.1f}\n"
                f"Pred age: {pred_age:.1f}"
            )

            records.append({
                "cam_method": "Grad-CAM++",
                "model": model_name,
                "task": task_type,
                "image_id": row["image_id"],
                "true_class": row["dx"],
                "predicted_class": None,
                "confidence": None,
                "true_age": row["age_target"],
                "predicted_age": float(pred_age)
            })

        axes[i, 0].imshow(display_img)
        axes[i, 0].set_title("Original Image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(cam_map)
        axes[i, 1].set_title("Grad-CAM++ Heatmap")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title(title_text)
        axes[i, 2].axis("off")

    plt.suptitle(f"{model_name} {task_type.capitalize()} Grad-CAM++", fontsize=16)
    plt.tight_layout()

    save_path = GRADCAMPP_DIR / f"{model_name.lower()}_{task_type}_gradcampp.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    campp.remove_hooks()

    del model
    del campp
    gc.collect()
    torch.cuda.empty_cache()

    print("Saved Grad-CAM++ figure:")
    print(save_path)

    return records

# ------------------------------------------------------------
# 9. Run Grad-CAM++ for all four models
# ------------------------------------------------------------
all_gradcampp_records = []

# 1. VGG16 Classification
records = run_gradcampp_for_model(
    model_name="VGG16",
    task_type="classification",
    model_builder=lambda: build_vgg16_classifier(num_classes),
    weights_path=MODEL_DIR / "vgg16_classification_best.pth",
    target_layer_getter=lambda model: model.features[28],
    selected_df=selected_df
)
all_gradcampp_records.extend(records)

# 2. VGG16 Regression
records = run_gradcampp_for_model(
    model_name="VGG16",
    task_type="regression",
    model_builder=build_vgg16_regressor,
    weights_path=MODEL_DIR / "vgg16_regression_best.pth",
    target_layer_getter=lambda model: model.features[28],
    selected_df=selected_df
)
all_gradcampp_records.extend(records)

# 3. ResNet50 Classification
records = run_gradcampp_for_model(
    model_name="ResNet50",
    task_type="classification",
    model_builder=lambda: build_resnet50_classifier(num_classes),
    weights_path=MODEL_DIR / "resnet50_classification_best.pth",
    target_layer_getter=lambda model: model.layer4[-1].conv3,
    selected_df=selected_df
)
all_gradcampp_records.extend(records)

# 4. ResNet50 Regression
records = run_gradcampp_for_model(
    model_name="ResNet50",
    task_type="regression",
    model_builder=build_resnet50_regressor,
    weights_path=MODEL_DIR / "resnet50_regression_best.pth",
    target_layer_getter=lambda model: model.layer4[-1].conv3,
    selected_df=selected_df
)
all_gradcampp_records.extend(records)

# ------------------------------------------------------------
# 10. Save Grad-CAM++ interpretation table
# ------------------------------------------------------------
gradcampp_df = pd.DataFrame(all_gradcampp_records)

gradcampp_csv_path = GRADCAMPP_DIR / "gradcampp_interpretation_records.csv"
gradcampp_df.to_csv(gradcampp_csv_path, index=False)

print("\nGrad-CAM++ interpretation records:")
display(gradcampp_df)

print("\nSaved Grad-CAM++ interpretation table:")
print(gradcampp_csv_path)

# ------------------------------------------------------------
# 11. Final status
# ------------------------------------------------------------
print("\nSTEP 11 COMPLETED SUCCESSFULLY.")
print("Bonus Grad-CAM++ visualizations generated for all four models.")
print("This satisfies the optional extra CAM variant bonus requirement.")